# How to Run 3D Gaussian Splatting (Windows)

## 1. 개요 (Overview)

이 문서는 Windows 환경에서 [3D Gaussian Splatting](https://github.com/graphdeco-inria/gaussian-splatting)을 설치하고 학습하는 전체 과정을 다룹니다.

**핵심 요구사항:**
- Windows 10/11
- NVIDIA GPU (CUDA 지원)
- Visual Studio 2022 (C++ 빌드 도구)
- **⭐ x64 Native Tools Command Prompt for VS 2022 필수**

---

## 2. 환경 설정 (Environment Setup)

### 2.1 Visual Studio 2022 설치

**⚠️ 중요: CUDA 확장 빌드를 위해 Visual Studio 2022 필수**

1. [Visual Studio 2022 Community](https://visualstudio.microsoft.com/downloads/) 다운로드
2. 설치 시 워크로드 선택:
   - ✅ **"Desktop development with C++"** (데스크톱 개발 C++)
   - ✅ Windows 10/11 SDK
   - ✅ MSVC v143 (또는 최신)
3. 설치 완료 후 **x64 Native Tools Command Prompt for VS 2022** 확인
   - 시작 메뉴 → Visual Studio 2022 → x64 Native Tools Command Prompt

### 2.2 CUDA Toolkit 설치

**원본 구현 권장: CUDA 11.8 | 테스트 버전: CUDA 12.4**

주의: 커스텀 CUDA 레이어가 있어 CUDA 버전에 따라 컴파일 실패 가능

**설치 링크:**
- [CUDA 11.8](https://developer.nvidia.com/cuda-11-8-0-download-archive)
- [CUDA 12.4](https://developer.nvidia.com/cuda-12-4-0-download-archive)

**설치 후 환경 변수 설정 (매 세션마다 필요):**

```cmd
REM ⭐ x64 Native Tools Command Prompt for VS 2022에서 실행
set CUDA_HOME=C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.4
set PATH=%CUDA_HOME%\bin;%CUDA_HOME%\libnvvp;%PATH%
set CUDA_PATH=%CUDA_HOME%
```



**CUDA 버전 확인:**

```cmd
nvcc --version
```



### 2.3 Python 환경 설정 (수동 방법 - 권장) ⭐

`environment.yml` 자동 설치가 실패하는 경우가 많아 **수동 설치 권장**

#### Step 1: x64 Native Tools Command Prompt 열기
시작 메뉴 → Visual Studio 2022 → **x64 Native Tools Command Prompt for VS 2022**

#### Step 2: 3DGS 저장소 클론

```cmd
git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
cd gaussian-splatting
```



#### Step 3: Conda 환경 생성

```cmd
conda create -n gaussian_splatting python=3.11 -y
conda activate gaussian_splatting
```



#### Step 4: CUDA 환경 변수 설정

```cmd
set CUDA_HOME=C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.4
set PATH=%CUDA_HOME%\bin;%CUDA_HOME%\libnvvp;%PATH%
set CUDA_PATH=%CUDA_HOME%
```



#### Step 5: PyTorch 설치 (CUDA 버전에 맞춰)

```cmd
REM CUDA 12.x용 (12.1, 12.4 모두 cu121 wheel 사용):
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

REM CUDA 11.8용:
REM pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
```



#### Step 6: 기본 패키지 설치

```cmd
pip install plyfile tqdm
```



#### Step 7: DISTUTILS 플래그 설정 (Windows 전용)

```cmd
set DISTUTILS_USE_SDK=1
```



#### Step 8: 커스텀 CUDA 확장 설치

```cmd
REM ⚠️ --no-build-isolation 플래그 필수
pip install --no-build-isolation submodules/diff-gaussian-rasterization
pip install --no-build-isolation submodules/simple-knn
```



**Step 8-1 (선택): Accelerated Rasterizer 설치 (Sparse Adam 사용 시)**

학습 속도 향상을 원하면 accelerated rasterizer 설치 (×1.6~×2.7 속도 향상):

```cmd
REM 1. 기존 rasterizer 제거
pip uninstall diff-gaussian-rasterization -y

REM 2. 빌드 캐시 삭제
cd submodules\diff-gaussian-rasterization
rmdir /s /q build

REM 3. 3dgs_accel 브랜치로 전환
git checkout 3dgs_accel

REM 4. Accelerated 버전 설치
pip install . --no-build-isolation

REM 5. 프로젝트 루트로 복귀
cd ..\..
```



이후 학습 시 `--optimizer_type sparse_adam` 옵션 사용 가능.

**성공 확인:**

```cmd
python -c "import diff_gaussian_rasterization; print('diff-gaussian-rasterization OK')"
python -c "import simple_knn; print('simple-knn OK')"
```



### 2.4 Alternative: WSL Ubuntu 설치 (Ubuntu 24.04 + CUDA 12.4)

Windows에서 Visual Studio 설정이 복잡하다면 **WSL (Windows Subsystem for Linux)**을 사용할 수 있습니다.

#### 2.4.1 WSL 및 Ubuntu 24.04 설치

```powershell
# PowerShell (관리자 권한)에서 실행
wsl --install -d Ubuntu-24.04
```



설치 후 재부팅하고 Ubuntu 사용자 계정 생성.

#### 2.4.2 CUDA Toolkit 12.4 설치 (WSL용)

**⚠️ Ubuntu 24.04 전용 패치 필요**

Ubuntu 24.04는 `libtinfo5` 의존성 문제로 CUDA 설치 전 수동 패치 필요:

In [ ]:
# 1. libtinfo5 의존성 패치 (Ubuntu 24.04 필수)
wget http://security.ubuntu.com/ubuntu/pool/universe/n/ncurses/libtinfo5_6.3-2ubuntu0.1_amd64.deb
sudo dpkg -i libtinfo5_6.3-2ubuntu0.1_amd64.deb
rm libtinfo5_6.3-2ubuntu0.1_amd64.deb

# 2. NVIDIA Repository 추가
wget https://developer.download.nvidia.com/compute/cuda/repos/wsl-ubuntu/x86_64/cuda-wsl-ubuntu.pin
sudo mv cuda-wsl-ubuntu.pin /etc/apt/preferences.d/cuda-repository-pin-600
wget https://developer.download.nvidia.com/compute/cuda/12.4.0/local_installers/cuda-repo-wsl-ubuntu-12-4-local_12.4.0-1_amd64.deb
sudo dpkg -i cuda-repo-wsl-ubuntu-12-4-local_12.4.0-1_amd64.deb
sudo cp /var/cuda-repo-wsl-ubuntu-12-4-local/cuda-*-keyring.gpg /usr/share/keyrings/

# 3. CUDA Toolkit 설치
sudo apt-get update
sudo apt-get -y install cuda-toolkit-12-4



#### 2.4.3 환경 변수 설정

`~/.bashrc` 파일 끝에 추가:

In [ ]:
export PATH=/usr/local/cuda-12.4/bin${PATH:+:${PATH}}
export LD_LIBRARY_PATH=/usr/local/cuda-12.4/lib64${LD_LIBRARY_PATH:+:${LD_LIBRARY_PATH}}



변경사항 적용:

In [ ]:
source ~/.bashrc
nvcc --version  # CUDA 12.4 확인



#### 2.4.4 빌드 도구 설치

**⚠️ 중요: GCC 12 필수 (Ubuntu 24.04 기본 GCC 13은 호환 안 됨)**

In [ ]:
# GLM 수학 라이브러리 + GCC 12 설치
sudo apt-get install build-essential libglm-dev g++-12 gcc-12

# 5-1. (선택) Accelerated Rasterizer 설치 (Sparse Adam 사용 시)
# 학습 속도 ×1.6~×2.7 향상
pip uninstall diff-gaussian-rasterization -y
cd submodules/diff-gaussian-rasterization
rm -rf build
git checkout 3dgs_accel
CC=gcc-12 CXX=g++-12 pip install . --no-build-isolation
cd ../..



#### 2.4.5 Python 환경 및 3DGS 설치

In [ ]:
# 1. 저장소 클론
git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
cd gaussian-splatting

# 2. Conda 환경 생성
conda create -n gaussian_splatting python=3.11 -y
conda activate gaussian_splatting

# 3. PyTorch 설치 (CUDA 12.x)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 4. 기본 패키지 설치
pip install plyfile tqdm

# 5. 커스텀 CUDA 확장 설치 (GCC 12 강제)
CC=gcc-12 CXX=g++-12 pip install ./submodules/diff-gaussian-rasterization --no-build-isolation
CC=gcc-12 CXX=g++-12 pip install ./submodules/simple-knn --no-build-isolation



**설치 확인:**

In [ ]:
pip list | grep gaussian
nvcc --version
python -c "import diff_gaussian_rasterization; print('Rasterizer OK')"
python -c "import simple_knn; print('KNN OK')"



**WSL 장점:**
- Visual Studio 설정 불필요
- Linux 네이티브 빌드 (더 안정적)
- 파일 시스템 성능 우수 (`/home/` 경로 사용 시)

**주의사항:**
- Windows 파일 시스템(`/mnt/c/`)은 느림 → WSL 내부(`~/`)에 데이터 저장 권장
- GPU 드라이버는 Windows 측에서 관리 (WSL 내부 설치 불필요)

---

## 3. 데이터 준비 (Data Preparation)

### 3.1 COLMAP 설치

3.10 dev 버전 사용했습니다.

**Option 1: Prebuilt Binary (권장)**
- [COLMAP Releases](https://github.com/colmap/colmap/releases)에서 Windows 빌드 다운로드
- 압축 해제 후 `COLMAP.bat` 실행 경로 확인

**Option 2: Conda 설치**

```cmd
conda install -c conda-forge colmap
```



### 3.2 convert.py로 COLMAP 실행

**입력 데이터 구조:**

```
vaultboy/
└── input/              # 여기에 원본 이미지 넣기 (.jpg, .png 등)
    ├── IMG_0001.jpg
    ├── IMG_0002.jpg
    └── ...
```



**COLMAP 실행:**

```cmd
REM ⭐ x64 Native Tools Command Prompt에서 실행
cd gaussian-splatting
python convert.py -s ../vaultboy --camera OPENCV
```



**주요 옵션:**
- `-s, --source_path`: 데이터셋 경로
- `--camera`: 카메라 모델 (OPENCV, PINHOLE, SIMPLE_PINHOLE)
- `--skip_matching`: feature matching 건너뛰기 (이미 COLMAP 결과 있을 때)
- `--colmap_executable`: COLMAP 실행 파일 경로 지정

### 3.3 convert.py 실행 후 폴더 구조

```
vaultboy/
├── input/                          # 원본 이미지 (사용자 준비)
├── images/                         # ⭐ Undistorted 이미지 (학습에 사용)
├── sparse/                         # ⭐ 최종 COLMAP 재구성 (학습에 사용)
│   └── 0/
│       ├── cameras.bin            # 카메라 내부 파라미터
│       ├── images.bin             # 카메라 외부 파라미터 (포즈)
│       └── points3D.bin           # 초기 3D 포인트 클라우드
├── distorted/                     # ⚙️ COLMAP 중간 결과물
│   ├── database.db               # Feature 매칭 DB
│   └── sparse/
│       └── 0/
│           ├── cameras.bin        # 렌즈 왜곡 포함된 카메라
│           ├── images.bin
│           ├── points3D.bin
│           └── project.ini
├── stereo/                        # COLMAP dense reconstruction용
├── run-colmap-geometric.sh
└── run-colmap-photometric.sh
```



**폴더 설명:**

| 폴더 | 설명 | 학습 사용 여부 |
|------|------|----------------|
| `input/` | 원본 이미지 (사용자가 준비) | ❌ |
| `images/` | **Undistorted 이미지** - 렌즈 왜곡 제거됨 | ✅ |
| `sparse/0/` | **최종 카메라 파라미터 + 3D 포인트** (PINHOLE/SIMPLE_PINHOLE) | ✅ |
| `distorted/` | **중간 결과물** - 원본 렌즈 왜곡 포함 (OPENCV 등) | ❌ |

**⭐ `distorted/` 폴더란?**

COLMAP은 다음 3단계로 작동합니다:
1. **Feature extraction + matching** → `distorted/database.db` 생성
2. **Bundle adjustment** → `distorted/sparse/0/` 생성 (렌즈 왜곡 파라미터 포함)
3. **Image undistortion** → `images/` + `sparse/0/` 생성 (왜곡 제거된 결과)

3D Gaussian Splatting은 **PINHOLE 또는 SIMPLE_PINHOLE 카메라 모델만 지원**하므로, `convert.py`가 자동으로 왜곡된 이미지를 이상적인 핀홀 카메라 이미지로 변환합니다.

- `distorted/sparse/0/cameras.bin`: OPENCV 모델 (radial distortion 파라미터 포함)
- `sparse/0/cameras.bin`: SIMPLE_PINHOLE 모델 (왜곡 없음, fx=fy)

**결론:** 학습 시에는 `images/` + `sparse/0/`만 사용되며, `distorted/`는 디버깅/재처리 목적으로 보관됩니다.

---

## 4. Depth Maps 생성 (선택 사항)

Depth regularization은 학습 품질을 크게 향상시킵니다. 자세한 내용은 [0403_depth_regularization.md](0403_depth_regularization.md) 참조.

### 4.1 Depth Anything V2 설치

```cmd
git clone https://github.com/DepthAnything/Depth-Anything-V2.git
cd Depth-Anything-V2
mkdir checkpoints
```



**가중치 다운로드:**
- [Depth-Anything-V2-Large](https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth)
- `checkpoints/depth_anything_v2_vitl.pth`로 저장

**의존성 설치:**

```cmd
pip install opencv-python matplotlib joblib huggingface_hub
```



### 4.2 Depth 맵 생성

```cmd
cd Depth-Anything-V2
python run.py --encoder vitl --pred-only --grayscale --img-path ../vaultboy/input --outdir ../vaultboy/depthmaps
```



**출력:** `vaultboy/depthmaps/` 폴더에 16-bit PNG depth 맵 생성

### 4.3 Depth Scale 정렬

Depth Anything V2의 상대 깊이를 COLMAP의 절대 스케일로 정렬:

```cmd
cd ../gaussian-splatting
python utils/make_depth_scale.py --base_dir ../vaultboy --depths_dir ../vaultboy/depthmaps
```



**출력:** `vaultboy/sparse/0/depth_params.json` 생성

**depth_params.json 예시:**

In [ ]:
{
  "IMG_0001": {
    "scale": 0.8542,
    "offset": 0.1234
  },
  "IMG_0002": {
    "scale": 0.8611,
    "offset": 0.1198
  }
}



**동작 원리:**
1. COLMAP sparse points를 각 이미지에 투영
2. 투영된 픽셀의 depth 값 추출
3. Monocular depth와 COLMAP depth를 median/MAD 기반으로 정렬
4. 각 이미지별 `scale`, `offset` 계산

---

## 5. 학습 (Training)

### 5.1 기본 학습 (Depth 없이)

```cmd
REM ⭐ x64 Native Tools Command Prompt에서 실행
conda activate gaussian_splatting
set CUDA_HOME=C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.4
set PATH=%CUDA_HOME%\bin;%PATH%

python train.py -s ../vaultboy
```



### 5.2 Depth Regularization 포함 학습

```cmd
python train.py -s ../vaultboy -d ../vaultboy/depthmaps --iterations 30000 --save_iterations 7000 15000 30000
```



### 5.3 고급 옵션 (최적화)

```cmd
python train.py -s ../vaultboy -d ../vaultboy/depthmaps ^
    --optimizer_type sparse_adam ^
    --exposure_lr_init 0.001 ^
    --exposure_lr_final 0.0001 ^
    --exposure_lr_delay_steps 5000 ^
    --exposure_lr_delay_mult 0.001 ^
    --train_test_exp ^
    --antialiasing
```



**주요 옵션:**
- `--optimizer_type sparse_adam`: 2.7배 학습 속도 향상 (메모리 효율적)
- `--exposure_lr_*`: Exposure compensation 학습률 (0402 문서 참조)
- `--train_test_exp`: 이미지 노출 보정 학습
- `--antialiasing`: 안티앨리어싱 활성화
- `--depth_ratio 0.01`: Depth loss 가중치 (기본값: 0.01)

**학습 출력:**

```
output/
└── <timestamp>/
    ├── point_cloud/
    │   └── iteration_30000/
    │       └── point_cloud.ply    # 최종 3D Gaussians
    ├── cameras.json
    └── cfg_args
```



---

## 6. 평가 및 시각화 (Evaluation)

### 6.1 온라인 뷰어 (SIBR Viewer)

**Prebuilt 뷰어 다운로드:**
- [Gaussian Splatting Releases](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/)
- `viewers/bin/SIBR_gaussianViewer_app.exe` 다운로드

**실행:**

```cmd
D:\GaussianSplattingTests\viewers\bin\SIBR_gaussianViewer_app.exe --m .\output\<timestamp>
```



### 6.2 렌더링 및 메트릭 계산

자세한 내용은 [0405_evaluation.md](0405_evaluation.md) 참조.

**1) Train/Test 분할 학습:**

```cmd
python train.py -s ../vaultboy --eval
```



**2) 테스트 뷰 렌더링:**

```cmd
python render.py -m output/<timestamp>
```


출력: `output/<timestamp>/test/ours_30000/renders/`

**3) 메트릭 계산:**

```cmd
python metrics.py -m output/<timestamp>
```


출력 메트릭: PSNR, SSIM, LPIPS

---

## 7. 문제 해결 (Troubleshooting)

### 7.1 CUDA 확장 빌드 실패

**증상:**

```
error: Microsoft Visual C++ 14.0 or greater is required
```



**해결:**
1. Visual Studio 2022 "Desktop development with C++" 워크로드 설치 확인
2. **⭐ x64 Native Tools Command Prompt 사용 확인**
3. `DISTUTILS_USE_SDK=1` 환경 변수 설정
4. CUDA 환경 변수 재설정

### 7.2 "Colmap camera model not handled" 오류

**증상:**

```
AssertionError: Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!
```



**원인:** `sparse/0/cameras.bin`이 OPENCV 등 왜곡 모델 포함

**해결:**

```cmd
python convert.py -s ../vaultboy --camera OPENCV --skip_matching
```


기존 `distorted/sparse/0/` 결과를 사용하여 undistortion만 재실행

### 7.3 Depth 학습 시 "depth_params.json not found"

**원인:** `make_depth_scale.py` 미실행

**해결:**

```cmd
python utils/make_depth_scale.py --base_dir ../vaultboy --depths_dir ../vaultboy/depthmaps
```



### 7.4 학습 중 Out of Memory (OOM)

**해결책:**
- 이미지 해상도 줄이기: `--resolution 2` (1/2 크기)
- 배치 크기 조정 (코드 수정 필요)
- `sparse_adam` 옵티마이저 사용

### 7.5 WSL/Ubuntu: "unrecognised option '--SiftExtraction.use_gpu'" 오류

**증상 (convert.py 실행 시):**

```
E20260123 22:36:50.809378 136971478605824 option_manager.cc:976] Failed to parse options - unrecognised option '--SiftExtraction.use_gpu'.
ERROR:root:Feature extraction failed with code 256. Exiting.
```



**원인:** COLMAP 3.9+ 버전에서 `--SiftExtraction.use_gpu` 옵션이 제거됨

**해결 방법 1: COLMAP 3.8 설치 (Conda, 권장)**

In [ ]:
# 호환되는 COLMAP 버전 설치
conda install -c conda-forge colmap=3.8



**해결 방법 2: convert.py 수정**

`convert.py` 파일에서 해당 라인 수정:

In [ ]:
# 수정 전 (Line ~70):
feat_extracton_cmd = colmap_command + " feature_extractor "\
    "--database_path " + database_path + " \
    --image_path " + args.source_path + "/input \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model " + args.camera + " \
    --SiftExtraction.use_gpu 1"

# 수정 후:
feat_extracton_cmd = colmap_command + " feature_extractor "\
    "--database_path " + database_path + " \
    --image_path " + args.source_path + "/input \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model " + args.camera
    # --SiftExtraction.use_gpu 제거 (COLMAP 3.9+에서 자동으로 GPU 사용)



**해결 방법 3: COLMAP 수동 실행**

`convert.py` 없이 COLMAP 직접 실행 후 `--skip_matching` 옵션 사용:

In [ ]:
# 1. Feature extraction
colmap feature_extractor \
    --database_path vaultboy/distorted/database.db \
    --image_path vaultboy/input \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model OPENCV

# 2. Feature matching
colmap exhaustive_matcher \
    --database_path vaultboy/distorted/database.db

# 3. Sparse reconstruction
mkdir -p vaultboy/distorted/sparse
colmap mapper \
    --database_path vaultboy/distorted/database.db \
    --image_path vaultboy/input \
    --output_path vaultboy/distorted/sparse

# 4. convert.py로 undistortion만 실행
python convert.py -s vaultboy --skip_matching



---

## 8. 추가 자료 (References)

**공식 저장소:**
- [3D Gaussian Splatting](https://github.com/graphdeco-inria/gaussian-splatting)
- [Depth Anything V2](https://github.com/DepthAnything/Depth-Anything-V2)
- [COLMAP](https://colmap.github.io/)

**논문:**
- [3D Gaussian Splatting for Real-Time Radiance Field Rendering (SIGGRAPH 2023)](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/)
- [Depth Anything V2 (NeurIPS 2024)](https://arxiv.org/abs/2406.09414)

**내부 문서:**
- [0402_exposure_compensation.md](0402_exposure_compensation.md)
- [0403_depth_regularization.md](0403_depth_regularization.md)
- [0404_simple_knn.md](0404_simple_knn.md)
- [0405_evaluation.md](0405_evaluation.md)

---

**마지막 업데이트:** 2026-01-23